In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os, gc
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import (LightningDataModule, LightningModule, Trainer)
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingRegressor

import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

import kaggle_evaluation.jane_street_inference_server

# CONFIG

In [2]:
class CONFIG:
    seed = 42
    target_col = "responder_6"
    # feature_cols = ["symbol_id", "time_id"] + [f"feature_{idx:02d}" for idx in range(79)]+ [f"responder_{idx}_lag_1" for idx in range(9)]
    feature_cols = [f"feature_{idx:02d}" for idx in range(79)]+ [f"responder_{idx}_lag_1" for idx in range(9)] + [f'symbol_id_{ind}' for ind in range(39)]
    
    model_paths = [
        # "/kaggle/input/js-with-lags-trained-xgb/result.pkl",
        # '/kaggle/input/yutong-nn-categorical'
        # '/kaggle/input/yutong-nn-large'
        '/kaggle/input/yutong-large-nn/pytorch/1/1'
    ]

# Load NN

In [3]:
# Custom R2 metric for validation
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2


class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))
            if i > 0:
                layers.append(nn.ReLU())
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            # layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))  # 输出层
        layers.append(nn.Tanh())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)  # 输出为一维张量

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w  # 考虑样本权重
        loss = loss.mean()
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        """Calculate validation WRMSE at the end of the epoch."""
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()
        if self.trainer.sanity_checking:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
        else:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
            weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()
            # r2_val
            val_r_square = r2_val(y, prob, weights)
            self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5,
                                                               verbose=True)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

In [4]:
N_folds = 5
models = []
for fold in range(N_folds):
    checkpoint_path = f"{CONFIG.model_paths[0]}/nn_{fold}.model.ckpt"
    model = NN.load_from_checkpoint(checkpoint_path)
    models.append(model.to("cuda:0"))

# CV score

In [5]:
valid = pl.scan_parquet(
    f"/kaggle/input/js24-preprocessing-create-lags/validation.parquet/"
).collect().to_pandas()

In [6]:
valid_new = pd.get_dummies(valid, columns=['symbol_id'])
valid_new

,id,date_id,time_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,responder_0,responder_1,responder_2,responder_3,responder_4,responder_5,responder_6,responder_7,responder_8,partition_id,label,responder_0_lag_1,responder_1_lag_1,responder_2_lag_1,responder_3_lag_1,responder_4_lag_1,responder_5_lag_1,responder_6_lag_1,responder_7_lag_1,responder_8_lag_1,symbol_id_0,symbol_id_1,symbol_id_2,symbol_id_3,symbol_id_4,symbol_id_5,symbol_id_6,symbol_id_7,symbol_id_8,symbol_id_9,symbol_id_10,symbol_id_11,symbol_id_12,symbol_id_13,symbol_id_14,symbol_id_15,symbol_id_16,symbol_id_17,symbol_id_18,symbol_id_19,symbol_id_20,symbol_id_21,symbol_id_22,symbol_id_23,symbol_id_24,symbol_id_25,symbol_id_26,symbol_id_27,symbol_id_28,symbol_id_29,symbol_id_30,symbol_id_31,symbol_id_32,symbol_id_33,symbol_id_34,symbol_id_35,symbol_id_36,symbol_id_37,symbol_id_38
0,46045114,1670,0,3.405333,0.310848,1.131483,0.745982,0.776556,-0.798888,0.189267,-0.661057,-0.290370,-0.228528,11,7,76,-0.721815,0.178626,-0.625948,NaN,-0.600020,NaN,-1.773468,-1.426673,0.752113,-0.128356,1.485508,0.798556,0.939105,0.531187,1.102988,0.510059,0.409313,-0.711646,-0.902749,-0.162899,NaN,NaN,0.619876,0.466504,-1.488351,0.000466,-0.099937,NaN,0.716869,NaN,NaN,0.108630,NaN,-0.957244,1.638018,-0.578405,0.482135,0.009865,NaN,0.258550,NaN,NaN,-0.701604,NaN,-1.752637,2.125332,NaN,0.074168,0.018479,1.027483,-0.525323,-0.239548,-0.619589,-1.701620,-1.398736,-0.939831,0.298140,-0.321387,-0.737133,0.032996,-0.614959,NaN,NaN,0.811054,0.708597,-0.001946,0.004254,-0.082806,-0.082336,-0.266573,-0.125054,1.400488,-1.104670,-0.119038,1.622135,-1.964823,9,0,-0.599179,-0.421537,-0.880075,0.227441,0.181244,-0.087156,0.213826,0.138748,0.242813,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,46045115,1670,0,2.619899,0.382185,0.458687,0.544353,0.092550,-0.911843,0.130969,-0.905152,-0.301003,-0.187945,11,7,76,-0.920349,2.437098,-0.075225,NaN,-0.504462,NaN,-1.682780,-1.309567,1.357302,-0.007474,1.107871,0.374563,1.059325,1.151365,-1.985968,-0.465673,0.825128,-0.756643,-0.735251,-0.006619,NaN,NaN,0.495858,0.553447,-1.584693,-0.058390,0.086217,NaN,0.514809,NaN,NaN,-0.652876,NaN,-1.598468,1.125729,-0.049231,0.421088,-0.185787,NaN,1.567681,NaN,NaN,-1.137877,NaN,-1.826585,1.230211,NaN,1.252662,0.437808,1.027483,-0.342516,-0.421323,-0.424025,-1.053866,-1.873317,-0.709916,1.826658,-0.019934,-0.572781,2.331671,-0.209500,NaN,NaN,0.859362,0.641228,-0.051073,-0.032598,0.195653,-0.018512,-0.006025,-0.244250,0.459814,-1.509297,-0.339291,0.753648,-1.822841,9,0,1.206133,0.458626,-0.101498,0.050520,0.018345,-0.553546,-1.088641,-0.553292,-1.762422,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,46045116,1670,0,2.213198,-0.163449,0.740222,0.272134,-0.092383,-1.215421,0.137453,-0.916160,-0.251347,-0.214681,81,2,59,-1.139882,-0.194705,-0.88

In [7]:
for feature_col in CONFIG.feature_cols:
    valid_new[feature_col] = valid_new[feature_col].ffill().fillna(0)
for feature_col in CONFIG.feature_cols:
    valid_new[feature_col] = valid_new[feature_col].astype('float32')
X_valid = valid_new[ CONFIG.feature_cols ]
y_valid = valid_new[ CONFIG.target_col ]
w_valid = valid_new[ "weight" ]
X_valid.shape, y_valid.shape, w_valid.shape

((1082224, 127), (1082224,), (1082224,))

In [8]:
models[0]

NN(
  (model): Sequential(
    (0): BatchNorm1d(127, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): Dropout(p=0.1, inplace=False)
    (2): Linear(in_features=127, out_features=512, bias=True)
    (3): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=512, out_features=512, bias=True)
    (7): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): Dropout(p=0.1, inplace=False)
    (10): Linear(in_features=512, out_features=512, bias=True)
    (11): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): Linear(in_features=512, out_features=256, bias=True)
    (14): Linear(in_features=256, out_features=1, bias=True)
    (15): Tanh()
  )
)

In [9]:
y_pred_valid = np.zeros(y_valid.shape)
with torch.no_grad():
    for model in models:
        model.eval()
        current_pred_valid = model(torch.FloatTensor(X_valid.to_numpy()).to("cuda:0")).cpu().numpy()
        current_r2 = r2_score( y_valid, current_pred_valid, sample_weight=w_valid )
        print(f'model r2 is {current_r2}')
        y_pred_valid += current_pred_valid / len(models)
valid_score = r2_score( y_valid, y_pred_valid, sample_weight=w_valid )
valid_score

model r2 is 0.10267110874705343
model r2 is 0.09572460353112733
model r2 is 0.09922584972518278
model r2 is 0.11517316472019135
model r2 is 0.04095095936945692


0.11524843854406719

# Online

In [147]:
# reload the model
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the pre-trained model
input_size = 127  # Replace with your feature count
output_size = 1  # Replace with your target count
checkpoint_path = f"{CONFIG.model_paths[0]}/nn_{4}.model.ckpt"
model = NN.load_from_checkpoint(checkpoint_path).to(device)

for layer in model.modules():
    if isinstance(layer, torch.nn.BatchNorm1d) or isinstance(layer, torch.nn.BatchNorm2d):
        layer.track_running_stats = False

Using device: cuda


In [148]:
y_pred_valid = np.zeros(y_valid.shape)
with torch.no_grad():
    model.eval()
    current_pred_valid = model(torch.FloatTensor(X_valid.to_numpy()).to("cuda:0")).cpu().numpy()
    current_r2 = r2_score( y_valid, current_pred_valid, sample_weight=w_valid )
    print(f'model r2 is {current_r2}')
    y_pred_valid += current_pred_valid 
valid_score = r2_score( y_valid, y_pred_valid, sample_weight=w_valid )
valid_score

model r2 is 0.04095095936945692


0.040950959161911826

In [149]:
import torch
import numpy as np
import random

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [150]:
import torch
from sklearn.metrics import r2_score
import torch.nn.functional as F

# Function to calculate R² score
def calculate_r2_score(model, data, feature_cols, target_col):
    """
    Predicts responder_6 and calculates R² score for a batch of rows.

    Args:
        model: Trained PyTorch model.
        data: DataFrame containing the rows to predict.
        feature_cols: List of feature column names.
        target_col: Name of the target column (responder_6).

    Returns:
        r2: R² score.
        predictions: Model predictions.
    """
    model.eval()  # Set model to evaluation mode
    features = torch.tensor(data[feature_cols].to_numpy(), dtype=torch.float32).to(device)
    true_values = data[target_col].to_numpy()
    with torch.no_grad():
        predictions = model(features).squeeze().cpu().numpy()
    r2 = r2_score(true_values, predictions, sample_weight=data['weight'].to_numpy())
    return r2, predictions

# Function to predict, calculate R², and update the model
def predict_update_cycle_with_overall_r2(model, optimizer, data, feature_cols, target_col, batch_size):
    """
    Enhanced prediction-update cycle with replay buffer to stabilize training.

    Args:
        model: Trained PyTorch model.
        optimizer: Optimizer for updating weights.
        criterion: Loss function.
        data: DataFrame containing all rows.
        feature_cols: List of feature column names.
        target_col: Name of the target column (responder_6).
        batch_size: Number of rows in each batch.
        replay_buffer_size: Maximum size of replay buffer.

    Returns:
        overall_r2: Overall R² score across all batches.
    """
    # from collections import deque
    # import random

    # replay_buffer = deque(maxlen=replay_buffer_size)  # Initialize replay buffer
    
    total_batches = len(data) // batch_size
    all_predictions = []  # Store predictions for overall R² calculation
    all_true_values = []  # Store true values for overall R² calculation
    all_weights = []      # Store weights for overall R² calculation

    for batch_idx in range(total_batches):
        # Select the current batch of rows
        start_idx = batch_idx * batch_size
        end_idx = start_idx + batch_size
        batch_data = data.iloc[start_idx:end_idx]
        
        # Step 1: Predict responder_6 and calculate R² for the batch
        r2, predictions = calculate_r2_score(model, batch_data, feature_cols, target_col)
        if batch_idx % 20 == 0:
            print(f"Batch {batch_idx + 1}/{total_batches} - R² Score: {r2}")

        # Accumulate predictions, true values, and weights
        all_predictions.extend(predictions)
        all_true_values.extend(batch_data[target_col].to_numpy())
        all_weights.extend(batch_data['weight'].to_numpy())  # Add weights for this batch

        # # Add current batch to replay buffer
        # for _, row in batch_data.iterrows():
        #     replay_buffer.append(row)

        # # Combine replay buffer and new batch for updates
        # if len(replay_buffer) >= batch_size:
        #     replay_sample = random.sample(replay_buffer, batch_size)
        #     replay_df = pd.DataFrame(replay_sample, columns=batch_data.columns)

        model.train()
        batch_features = torch.tensor(batch_data[feature_cols].values, dtype=torch.float32).to(device)
        batch_targets = torch.tensor(batch_data[target_col].values, dtype=torch.float32).unsqueeze(1).to(device)
        batch_weights = torch.tensor(batch_data['weight'].values, dtype=torch.float32).to(device)

        optimizer.zero_grad()
        batch_predictions = model(batch_features)
        loss = F.mse_loss(batch_predictions, batch_targets) * batch_weights
        loss = loss.mean()
        loss.backward()
        optimizer.step()
        if batch_idx % 20 == 0:
            print(f"Batch {batch_idx + 1}/{total_batches} - Loss: {loss.item()}")

    # Step 3: Calculate and report overall R² score
    overall_r2 = r2_score(all_true_values, all_predictions, sample_weight=all_weights)
    print(f"Overall R² Score: {overall_r2}")

    return overall_r2

In [151]:
# Run the predict-update cycle and report the overall R² score
overall_r2 = predict_update_cycle_with_overall_r2(
    model=model,
    optimizer=optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-4),
    data=valid_new,
    feature_cols=CONFIG.feature_cols,
    target_col='responder_6',
    batch_size=4096
)

print(f"Final Overall R² Score: {overall_r2}")


Batch 1/264 - R² Score: -0.11802558457206636
Batch 1/264 - Loss: 2.4741032123565674
Batch 21/264 - R² Score: 0.027087876583881343
Batch 21/264 - Loss: 0.5806628465652466
Batch 41/264 - R² Score: -0.00991243501747996
Batch 41/264 - Loss: 0.8699268102645874
Batch 61/264 - R² Score: 0.0024007325425701165
Batch 61/264 - Loss: 0.8900855779647827
Batch 81/264 - R² Score: 0.025127046630011307
Batch 81/264 - Loss: 2.1665289402008057
Batch 101/264 - R² Score: 0.02822607511181041
Batch 101/264 - Loss: 2.1779332160949707
Batch 121/264 - R² Score: 0.01972306631510945
Batch 121/264 - Loss: 0.9262826442718506
Batch 141/264 - R² Score: -0.013116590253433813
Batch 141/264 - Loss: 1.7536334991455078
Batch 161/264 - R² Score: -0.0026309798616355184
Batch 161/264 - Loss: 0.5543968081474304
Batch 181/264 - R² Score: -0.0020926411912909337
Batch 181/264 - Loss: 1.205779790878296
Batch 201/264 - R² Score: 0.052515425916987324
Batch 201/264 - Loss: 2.5168724060058594
Batch 221/264 - R² Score: 0.0018006939767

# Submission

In [ ]:
lags_ : pl.DataFrame | None = None
    
def predict(test: pl.DataFrame, lags: pl.DataFrame | None) -> pl.DataFrame | pd.DataFrame:
    global lags_
    if lags is not None:
        lags_ = lags

    predictions = test.select(
        'row_id',
        pl.lit(0.0).alias('responder_6'),
    )
    symbol_ids = test.select('symbol_id').to_numpy()[:, 0]

    if not lags is None:
        lags = lags.group_by(["date_id", "symbol_id"], maintain_order=True).last() # pick up last record of previous date
        test = test.join(lags, on=["date_id", "symbol_id"],  how="left")
    else:
        test = test.with_columns(
            ( pl.lit(0.0).alias(f'responder_{idx}_lag_1') for idx in range(9) )
        )
    

    print(test.shape)
    
    # one-hot encoding of symbol_id
    symbol_ids = test.select(pl.col("symbol_id").unique().sort()).to_series()
    # Create dummy variables for each category
    test = test.with_columns([
        (pl.col("symbol_id") == idx).cast(pl.UInt8).alias(f"symbol_id_{idx}")
        for idx in range(39)
    ]).drop("symbol_id")  # Drop the original 'category' column


    print(test.shape)
    
    preds = np.zeros((test.shape[0],))
    test_input = test[CONFIG.feature_cols].to_pandas()

    print(test_input.shape)

    
    test_input = test_input.fillna(method = 'ffill').fillna(0)
    test_input = torch.FloatTensor(test_input.values).to("cuda:0")

    print(test_input.shape)

    with torch.no_grad():
        for i, nn_model in enumerate(tqdm(models)):
            nn_model.eval()
            preds += nn_model(test_input).cpu().numpy() / len(models)
    print(f"predict> preds.shape =", preds.shape)
    
    predictions = \
    test.select('row_id').\
    with_columns(
        pl.Series(
            name   = 'responder_6', 
            values = np.clip(preds, a_min = -5, a_max = 5),
            dtype  = pl.Float64,
        )
    )

    # The predict function must return a DataFrame
    assert isinstance(predictions, pl.DataFrame | pd.DataFrame)
    # with columns 'row_id', 'responer_6'
    assert list(predictions.columns) == ['row_id', 'responder_6']
    # and as many rows as the test data.
    assert len(predictions) == len(test)

    return predictions

In [ ]:
inference_server = kaggle_evaluation.jane_street_inference_server.JSInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        (
            '/kaggle/input/jane-street-realtime-marketdata-forecasting/test.parquet',
            '/kaggle/input/jane-street-realtime-marketdata-forecasting/lags.parquet',
        )
    )